# Architecture Understanding — Visualizing What GNNs Learn on QM9

This notebook produces report-ready figures that reveal **how** each GNN
architecture (GCN, GAT, SchNet) processes molecular graphs differently.

**Requires:** Trained checkpoints for GCN, GAT, and SchNet in
`outputs/checkpoints/`. Run `train_colab.ipynb` first.

**Figures produced:**
We include both **GAT** and **GATv2** because GATv2 incorporates bond-type
edge attributes into its attention scores (via `edge_dim`), while standard GAT
computes attention from node features alone. Comparing them isolates the effect
of edge-aware attention on learned representations.

1. t-SNE of learned graph embeddings (3 panels)
2. GAT attention weight visualization on example molecules
3. Layer-wise embedding similarity (over-smoothing diagnostic)
4. Prediction error vs molecule size
5. Predicted vs true scatter plots
6. All-models training curve comparison

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DRIVE_ROOT  = '/content/drive/MyDrive/gnn_qm9'
PROJECT_DIR = '/content/gnn_qm9'
REPO_URL    = 'https://github.com/amanikonda123/DL-Final-Project.git'

os.makedirs(f'{DRIVE_ROOT}/outputs/plots/analysis', exist_ok=True)

if os.path.exists(PROJECT_DIR):
    !git -C {PROJECT_DIR} pull
else:
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = 'cu121' if torch.cuda.is_available() else 'cpu'
!pip install -q torch-geometric
!pip install -q pyg-lib torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html
!pip install -q pyyaml tqdm scikit-learn

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT  = f'{DRIVE_ROOT}/data/qm9_raw'
OUTPUT_DIR = f'{DRIVE_ROOT}/outputs'
PLOT_DIR   = f'{OUTPUT_DIR}/plots/analysis'
print(f'Device: {DEVICE}')

In [ ]:
# ── Cell 2: Load Trained Models and Data ─────────────────────────────────────
from data.loader import get_dataloaders
from data.features import select_features, get_feature_dims
from models import build_model

# Load configs
with open("config/gcn.yaml") as f: gcn_cfg = yaml.safe_load(f)
with open("config/gat.yaml") as f: gat_cfg = yaml.safe_load(f)
with open("config/schnet.yaml") as f: schnet_cfg = yaml.safe_load(f)

# Load data
topo_train, topo_val, topo_test, topo_norm = get_dataloaders(gcn_cfg, root=DATA_ROOT)
geo_train, geo_val, geo_test, geo_norm = get_dataloaders(schnet_cfg, root=DATA_ROOT)

# Build and load models
def load_model(name, cfg, ckpt_path):
    feature_dims = get_feature_dims(cfg["dataset"].get("feature_mode", "topology"))
    model = build_model(name, cfg, feature_dims).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    model.eval()
    param_count = sum(p.numel() for p in model.parameters())
    print(f"  {name.upper()}: loaded ({param_count:,} params)")
    return model

print("Loading checkpoints...")
gcn_model    = load_model("gcn",    gcn_cfg,    f"{OUTPUT_DIR}/checkpoints/gcn_best.pt")
gat_model    = load_model("gat",    gat_cfg,    f"{OUTPUT_DIR}/checkpoints/gat_best.pt")
schnet_model = load_model("schnet", schnet_cfg, f"{OUTPUT_DIR}/checkpoints/schnet_best.pt")
print("All models loaded.")

## Figure 1: t-SNE of Learned Graph Embeddings

Extract the graph-level embedding (before the regression head) from each model
on ~5000 test molecules. Run t-SNE and color by the true property value.

This reveals how each architecture organizes molecular space. SchNet should
show smoother property gradients since it uses 3D atomic coordinates.

In [ ]:
# ── Cell 3: t-SNE of Graph Embeddings ────────────────────────────────────────
N_SAMPLES = 5000

def collect_embeddings(model, loader, normalizer, model_name, feature_mode, n=N_SAMPLES):
    """Extract graph-level embeddings and true targets for n test molecules."""
    embeddings, targets = [], []
    count = 0
    with torch.no_grad():
        for batch in loader:
            batch = select_features(batch, feature_mode)
            batch = batch.to(DEVICE)
            if model_name == "schnet":
                emb, _ = model.get_embedding(batch.z, batch.pos, batch.batch)
            else:
                emb, _ = model.get_embedding(batch.x, batch.edge_index, batch.batch)
            embeddings.append(emb.cpu())
            targets.append(normalizer.denormalize(batch.y.cpu()))
            count += batch.num_graphs
            if count >= n:
                break
    return torch.cat(embeddings)[:n].numpy(), torch.cat(targets)[:n].numpy()

print("Extracting embeddings...")
gcn_emb,    gcn_tgt    = collect_embeddings(gcn_model,    topo_test, topo_norm, "gcn",    "topology")
gat_emb,    gat_tgt    = collect_embeddings(gat_model,    topo_test, topo_norm, "gat",    "topology")
schnet_emb, schnet_tgt = collect_embeddings(schnet_model, geo_test,  geo_norm,  "schnet", "geometry")

print("Running t-SNE (this may take a minute)...")
gcn_tsne    = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(gcn_emb)
gat_tsne    = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(gat_emb)
schnet_tsne = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(schnet_emb)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("t-SNE of Learned Graph Embeddings — Colored by True U0 (Ha)", fontsize=14, y=1.02)

for ax, tsne, tgt, title in [
    (axes[0], gcn_tsne,    gcn_tgt,    "GCN"),
    (axes[1], gat_tsne,    gat_tgt,    "GAT"),
    (axes[2], schnet_tsne, schnet_tgt, "SchNet"),
]:
    sc = ax.scatter(tsne[:, 0], tsne[:, 1], c=tgt.ravel(), cmap="viridis",
                    s=4, alpha=0.6, rasterized=True)
    ax.set_title(title, fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])

cbar = fig.colorbar(sc, ax=axes, shrink=0.8, pad=0.02)
cbar.set_label("True U0 (Ha)", fontsize=11)

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/tsne_embeddings.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {PLOT_DIR}/tsne_embeddings.png")

## Figure 2: GAT Attention Weight Visualization

Extract attention coefficients from GATConv layers on individual molecules.
Shows which atom pairs the attention mechanism focuses on.

In [ ]:
# ── Cell 4: GAT Attention Visualization ──────────────────────────────────────
import networkx as nx
from torch_geometric.utils import to_networkx

ATOM_LABELS = {1: "H", 6: "C", 7: "N", 8: "O", 9: "F"}

# Pick 4 diverse molecules from the test set
test_mols = []
for batch in topo_test:
    for i in range(batch.num_graphs):
        from torch_geometric.utils import subgraph
        mask = batch.batch == i
        mol_data = batch.__class__()
        mol_data.x = batch.x[mask]
        mol_data.z = batch.z[mask]
        mol_data.edge_index = subgraph(mask, batch.edge_index, relabel_nodes=True)[0]
        mol_data.y = batch.y[i:i+1]
        mol_data.batch = torch.zeros(mask.sum().item(), dtype=torch.long)
        test_mols.append(mol_data)
        if len(test_mols) >= 200:
            break
    if len(test_mols) >= 200:
        break

# Select molecules of varying sizes
sizes = [m.x.size(0) for m in test_mols]
idx_small  = min(range(len(sizes)), key=lambda i: abs(sizes[i] - 5))
idx_medium = min(range(len(sizes)), key=lambda i: abs(sizes[i] - 12))
idx_large  = min(range(len(sizes)), key=lambda i: abs(sizes[i] - 18))
idx_xl     = min(range(len(sizes)), key=lambda i: abs(sizes[i] - 25))
selected = [test_mols[i] for i in [idx_small, idx_medium, idx_large, idx_xl]]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("GAT Attention Weights — Last Layer (edge thickness = attention)", fontsize=14, y=1.02)

for ax, mol in zip(axes, selected):
    mol_dev = mol.to(DEVICE)
    mol_dev.x = select_features(mol_dev, "topology").x
    attn_list = gat_model.get_attention_weights(mol_dev.x, mol_dev.edge_index, mol_dev.batch)
    ei, alpha = attn_list[-1]  # last layer
    alpha_mean = alpha.mean(dim=-1).cpu().numpy()  # average over heads

    G = nx.Graph()
    n_nodes = mol.x.size(0)
    for n in range(n_nodes):
        G.add_node(n)

    ei_cpu = ei.cpu().numpy()
    edge_weights = {}
    for e in range(ei_cpu.shape[1]):
        src, dst = int(ei_cpu[0, e]), int(ei_cpu[1, e])
        if src < dst:
            key = (src, dst)
            edge_weights[key] = edge_weights.get(key, 0) + alpha_mean[e]
    for (u, v), w in edge_weights.items():
        G.add_edge(u, v, weight=w)

    pos = nx.spring_layout(G, seed=42)
    labels = {n: ATOM_LABELS.get(mol.z[n].item(), "?") for n in range(n_nodes)}
    weights = [G[u][v]["weight"] for u, v in G.edges()]
    max_w = max(weights) if weights else 1
    widths = [3.0 * w / max_w + 0.5 for w in weights]

    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=300, node_color="lightblue", edgecolors="black")
    nx.draw_networkx_edges(G, pos, ax=ax, width=widths, alpha=0.7, edge_color="steelblue")
    nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=8)
    ax.set_title(f"{n_nodes} atoms", fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/gat_attention.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {PLOT_DIR}/gat_attention.png")

## Figure 3: Layer-wise Embedding Similarity (Over-smoothing)

Compute the mean pairwise cosine similarity of node embeddings after each GNN layer.
If similarity approaches 1.0, all nodes become indistinguishable (over-smoothing).
GAT's attention should resist this better than GCN's uniform aggregation.

In [ ]:
# ── Cell 5: Over-smoothing Diagnostic ────────────────────────────────────────
from torch.nn.functional import cosine_similarity

def compute_layer_similarity(model, loader, model_name, feature_mode, n_batches=10):
    """Compute mean pairwise cosine similarity per layer."""
    layer_sims = None
    count = 0

    with torch.no_grad():
        for batch in loader:
            batch = select_features(batch, feature_mode)
            batch = batch.to(DEVICE)
            if model_name == "schnet":
                break  # SchNet does not expose per-layer node embeddings
            _, layer_embeds = model.get_embedding(batch.x, batch.edge_index, batch.batch)

            if layer_sims is None:
                layer_sims = [0.0] * len(layer_embeds)

            for li, emb in enumerate(layer_embeds):
                emb_norm = emb / (emb.norm(dim=1, keepdim=True) + 1e-8)
                sim_matrix = emb_norm @ emb_norm.T
                n = emb_norm.size(0)
                # Mean of off-diagonal elements
                mask = ~torch.eye(n, dtype=torch.bool, device=sim_matrix.device)
                mean_sim = sim_matrix[mask].mean().item()
                layer_sims[li] += mean_sim

            count += 1
            if count >= n_batches:
                break

    return [s / count for s in layer_sims] if layer_sims else []

print("Computing layer-wise similarity...")
gcn_sims = compute_layer_similarity(gcn_model, topo_test, "gcn", "topology")
gat_sims = compute_layer_similarity(gat_model, topo_test, "gat", "topology")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(gcn_sims)), gcn_sims, "o-", label="GCN", linewidth=2, markersize=7)
ax.plot(range(len(gat_sims)), gat_sims, "s-", label="GAT", linewidth=2, markersize=7)
ax.axhline(y=1.0, color="red", linestyle="--", alpha=0.5, label="Perfect similarity")
ax.set_xlabel("Layer Index", fontsize=12)
ax.set_ylabel("Mean Pairwise Cosine Similarity", fontsize=12)
ax.set_title("Over-smoothing Diagnostic — Node Embedding Similarity per Layer", fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/oversmoothing.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {PLOT_DIR}/oversmoothing.png")

## Figure 4: Prediction Error vs Molecule Size

Are larger molecules harder to predict? Topology-based models may struggle more
with bigger molecules because they miss geometric interactions, while SchNet
should maintain accuracy regardless of size.

In [ ]:
# ── Cell 6: Error vs Molecule Size ───────────────────────────────────────────
from utils.metrics import mae as compute_mae

def collect_errors_by_size(model, loader, normalizer, model_name, feature_mode):
    """Collect (num_atoms, abs_error) for each test molecule."""
    sizes, errors = [], []
    with torch.no_grad():
        for batch in loader:
            batch_proc = select_features(batch, feature_mode)
            batch_proc = batch_proc.to(DEVICE)

            if model_name == "schnet":
                pred = model(batch_proc.z, batch_proc.pos, batch_proc.batch)
            else:
                pred = model(batch_proc.x, batch_proc.edge_index, batch_proc.batch)

            pred_de = normalizer.denormalize(pred.cpu())
            tgt_de  = normalizer.denormalize(batch_proc.y.cpu())

            for i in range(batch.num_graphs):
                n_atoms = (batch.batch == i).sum().item()
                err = abs(pred_de[i].item() - tgt_de[i].item())
                sizes.append(n_atoms)
                errors.append(err)
    return np.array(sizes), np.array(errors)

print("Collecting prediction errors by molecule size...")
gcn_sizes,    gcn_errors    = collect_errors_by_size(gcn_model,    topo_test, topo_norm, "gcn",    "topology")
gat_sizes,    gat_errors    = collect_errors_by_size(gat_model,    topo_test, topo_norm, "gat",    "topology")
schnet_sizes, schnet_errors = collect_errors_by_size(schnet_model, geo_test,  geo_norm,  "schnet", "geometry")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Prediction Error vs Molecule Size (Number of Atoms)", fontsize=14, y=1.02)

for ax, sizes, errors, title in [
    (axes[0], gcn_sizes,    gcn_errors,    "GCN"),
    (axes[1], gat_sizes,    gat_errors,    "GAT"),
    (axes[2], schnet_sizes, schnet_errors, "SchNet"),
]:
    ax.scatter(sizes, errors, s=3, alpha=0.3, rasterized=True)
    # Trend line (binned mean)
    unique_sizes = sorted(set(sizes))
    bin_means = [np.mean(errors[sizes == s]) for s in unique_sizes]
    ax.plot(unique_sizes, bin_means, "r-", linewidth=2, label="Mean error")
    ax.set_xlabel("Number of Atoms", fontsize=11)
    ax.set_ylabel("Absolute Error (Ha)", fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/error_vs_size.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {PLOT_DIR}/error_vs_size.png")

## Figure 5: Predicted vs True Property Values

Points on the diagonal indicate perfect predictions. Systematic deviations
reveal where each model struggles.

In [ ]:
# ── Cell 7: Predicted vs True ────────────────────────────────────────────────
def collect_predictions(model, loader, normalizer, model_name, feature_mode):
    """Collect (predicted, true) for all test molecules."""
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = select_features(batch, feature_mode)
            batch = batch.to(DEVICE)
            if model_name == "schnet":
                pred = model(batch.z, batch.pos, batch.batch)
            else:
                pred = model(batch.x, batch.edge_index, batch.batch)
            preds.append(normalizer.denormalize(pred.cpu()))
            targets.append(normalizer.denormalize(batch.y.cpu()))
    return torch.cat(preds).numpy(), torch.cat(targets).numpy()

print("Collecting predictions...")
gcn_pred,    gcn_true    = collect_predictions(gcn_model,    topo_test, topo_norm, "gcn",    "topology")
gat_pred,    gat_true    = collect_predictions(gat_model,    topo_test, topo_norm, "gat",    "topology")
schnet_pred, schnet_true = collect_predictions(schnet_model, geo_test,  geo_norm,  "schnet", "geometry")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Predicted vs True U0 — Test Set", fontsize=14, y=1.02)

for ax, pred, true, title in [
    (axes[0], gcn_pred,    gcn_true,    "GCN"),
    (axes[1], gat_pred,    gat_true,    "GAT"),
    (axes[2], schnet_pred, schnet_true, "SchNet"),
]:
    err = np.abs(pred.ravel() - true.ravel())
    sc = ax.scatter(true.ravel(), pred.ravel(), c=err, cmap="hot_r",
                    s=3, alpha=0.5, rasterized=True)
    lims = [min(true.min(), pred.min()), max(true.max(), pred.max())]
    ax.plot(lims, lims, "k--", linewidth=1, alpha=0.5)
    ax.set_xlabel("True U0 (Ha)", fontsize=11)
    ax.set_ylabel("Predicted U0 (Ha)", fontsize=11)
    ax.set_title(f"{title}  |  MAE={np.mean(err):.2f}", fontsize=12)
    ax.grid(alpha=0.3)

cbar = fig.colorbar(sc, ax=axes, shrink=0.8, pad=0.02)
cbar.set_label("Absolute Error (Ha)", fontsize=11)

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/predicted_vs_true.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {PLOT_DIR}/predicted_vs_true.png")

## Figure 6: All-Models Training Curve Comparison

Overlay training and validation curves for all models.
Shows relative convergence speed and final performance.

In [ ]:
# ── Cell 8: Training Curves Comparison ───────────────────────────────────────
import glob

history_files = glob.glob(f"{OUTPUT_DIR}/logs/*_history.csv")
histories = {}
for path in sorted(history_files):
    name = os.path.basename(path).replace("_history.csv", "")
    try:
        df = pd.read_csv(path)
        if len(df) > 0:
            histories[name] = df
            print(f"  {name}: {len(df)} epochs")
    except Exception:
        pass

if len(histories) == 0:
    print("No training histories found. Run train_colab.ipynb first.")
else:
    COLORS = {"gcn": "#4C72B0", "gat": "#DD8452", "gatv2": "#55A868", "schnet": "#C44E52"}

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle("Training Curves — All Models", fontsize=14)

    for name, df in histories.items():
        color = COLORS.get(name, "grey")
        axes[0].plot(df["epoch"], df["val_loss"], label=f"{name.upper()} val",
                     color=color, linewidth=2)
        axes[0].plot(df["epoch"], df["train_loss"], label=f"{name.upper()} train",
                     color=color, linewidth=1, linestyle="--", alpha=0.5)

        axes[1].plot(df["epoch"], df["val_mae"], label=name.upper(),
                     color=color, linewidth=2)

    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("MSE Loss (normalized)")
    axes[0].set_title("Train/Val Loss")
    axes[0].legend(fontsize=8, ncol=2)
    axes[0].grid(alpha=0.3)

    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Val MAE (Ha)")
    axes[1].set_title("Validation MAE")
    axes[1].legend(fontsize=10)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/training_curves_all.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {PLOT_DIR}/training_curves_all.png")

In [ ]:
# ── Cell 9: Summary ──────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("ALL FIGURES GENERATED")
print("=" * 60)
print(f"\nSaved to: {PLOT_DIR}/")
print()
for f in sorted(os.listdir(PLOT_DIR)):
    if f.endswith(".png"):
        print(f"  {f}")
print()
print("These figures are ready to include in your report.")